In [21]:
import pandas as pd
import numpy as np
from pandas import DataFrame, Series
import sys          # only needed to determine Python version number

# Set some pandas parameters..
#pd.set_option('display.notebook_repr_html', True)
#pd.set_option('display.max_columns', 10)
#pd.set_option('display.max_rows', 10)

# Matplotlib ptions
import matplotlib.pyplot as plt
import matplotlib   # only needed to determine Matplotlib version number
# Render our plots inline:
%matplotlib inline  
pd.set_option('display.mpl_style', 'default') # Make the graphs a bit prettier
plt.rcParams['figure.figsize'] = (15, 5)

#print('Python version ' + sys.version)
#print('Pandas version ' + pd.__version__)
#print('Matplotlib version ' + matplotlib.__version__)
#pd.show_versions()


In [22]:
scenarios = ['P0-Mi-Ei'] # for quick testing
#scenarios = ['P0-Mh-Ei', 'P1-Mh-Ei', 'P0-Mi-Ei', 'P1-Mi-Ei', 'P0-Me-Ei', 'P1-Me-Ei', 'P0-Mi-E0', 'P1-Mi-E0', 'P0-Mi-E1', 'P1-Mi-E1']
baseIDir = 'output/results'

productData_byscenario = [pd.read_csv(baseIDir+'/productData_'+scenario+'.csv',sep=';',  dtype={'value': np.float64}) for scenario in scenarios]
productDataReg = pd.concat(productData_byscenario)
productDataReg = productDataReg.drop(['Unnamed: 8'],axis=1) # we don't want this column never
#productDataReg['prodLev'] = np.where(productDataReg['prod'] in ['hardWRoundW', 'softWRoundW', 'pulpWFuelW'], 1, 2)
productDataReg['prodLev'] = np.where(productDataReg['prod'].isin(['hardWRoundW', 'softWRoundW', 'pulpWFuelW','ashRoundW']), 'primPr', 'secPr')
productDataReg = productDataReg.drop(['freeDim'],axis=1) # we don't want this column never
productDataReg = productDataReg[productDataReg['parName'] != 'rt'] # we don't want rt in national data
productDataReg.set_index(['scen','country','region','prod','prodLev','year','parName'], inplace=True)
productDataReg=productDataReg.unstack(['parName'])

lagged_productDataReg = productDataReg.groupby(level=['scen','country','region','prod','prodLev']).shift()




productDataReg[90:110]







value                      \
parName                                               dl        da        dt   
scen     country region prod      prodLev year                                 
P0-Mi-Ei 11000   11010  ashPlyW   secPr   2095  0.005935  0.008194  0.014129   
                                          2096  0.005936  0.008194  0.014130   
                                          2097  0.005937  0.008194  0.014131   
                                          2098  0.005937  0.008194  0.014132   
                                          2099  0.005938  0.008195  0.014132   
                                          2100  0.005938  0.008195  0.014133   
                                          2101  0.005938  0.008195  0.014133   
                                          2102  0.005938  0.008195  0.014133   
                                          2103  0.005938  0.008195  0.014134   
                                          2104  0.005938  0.008195  0.014134   
                                          2105  0.005939  0.008196  0.014134   
                                          2106  0.005939  0.008196  0.014135   
                                          2107  0.005939  0.008196  0.014135   
                        ashRoundW primPr  2005  0.000303  0.000000  0.000303   
                                          2006  0.000303  0.000000  0.000303   
                                          2007  0.000534  0.000000  0.000534   
                                          2008  0.000587  0.000000  0.000587   
                                          2009  0.000646  0.000000  0.000646   
                                          2010  0.000711  0.000000  0.000711   
                                          2011  0.000782  0.000000  0.000782   

                                                                              \
parName                                               dc        sl        sa   
scen     country region prod      prodLev year                                 
P0-Mi-Ei 11000   11010  ashPlyW   secPr   2095  0.007402  0.000000  0.000000   
                                          2096  0.007403  0.000000  0.000000   
                                          2097  0.007403  0.000000  0.000000   
                                          2098  0.007403  0.000000  0.000000   
                                          2099  0.007404  0.000000  0.000000   
                                          2100  0.007404  0.000000  0.000000   
                                          2101  0.007404  0.000000  0.000000   
                                          2102  0.007404  0.000000  0.000000   
                                          2103  0.007404  0.000000  0.000000   
                                          2104  0.007405  0.000000  0.000000   
                                          2105  0.007405  0.000000  0.000000   
                                          2106  0.007405  0.000000  0.000000   
                                          2107  0.007405  0.000000  0.000000   
                        ashRoundW primPr  2005  0.000000  0.012059  0.002225   
                                          2006  0.000000  0.012059  0.002225   
                                          2007  0.000000  0.011299  0.002398   
                                          2008  0.000000  0.011272  0.002448   
                                          2009  0.000000  0.011103  0.001911   
                                          2010  0.000000  0.011044  0.002014   
                                          2011  0.000000  0.011056  0.002271   

                                                                              \
parName                                               st        sc        pl   
scen     country region prod      prodLev year                                 
P0-Mi-Ei 11000   11010  ashPlyW   secPr   2095  0.000000  0.000000  609.1190   
                                          2096  0

## Computing producer and consumer surpluses ##

### Producer surplus ###
$ S_t = S_{t-1} P_{t-1}^{-\sigma} * {\left(  \frac{In_t}{In_{t-1}} \right)}^ {\frac{\gamma}{\sigma} } * P_{t}^{\sigma}  $

$ P_t = P _{t-1} * S_{t-1}^{-\frac{1}{\sigma}}* {\left(  \frac{In_{t-1}}{In_t} \right)}^ {\frac{\gamma}{\sigma} } * S_t^{\frac{1}{\sigma}} $

$ \int P(S) ~ dS = P _{t-1} * S_{t-1}^{-\frac{1}{\sigma}}* {\left(  \frac{In_{t-1}}{In_t} \right)}^ {\frac{\gamma}{\sigma} } * S_{t}^{\frac{\sigma+1}{\sigma}} *  \frac{\sigma}{\sigma+1} $

Producer surplus =  $ P^* * S^* - P _{t-1} * S_{t-1}^{-\frac{1}{\sigma}}* {\left(  \frac{In_{t-1}}{In_t} \right)}^ {\frac{\gamma}{\sigma} } * S^{* ~ \frac{\sigma+1}{\sigma}} *  \frac{\sigma}{\sigma+1}    $

### Consumer surplus ###

$ D_t = D_{t-1} * P_{t-1}^{-\sigma}*P_t^\sigma$

$P_t = P_{t-1} * D_{t-1}^{-\frac{1}{\sigma}} * D_t^\frac{1}{\sigma} $

$ \int P(D) ~ dD = P_{t-1} * D_{t-1}^{-\frac{1}{\sigma}} * D_t^\frac{\sigma+1}{\sigma} * \frac{\sigma}{\sigma +1 }$

Consumer surplus = $ \left[ P_{t-1} * D_{t-1}^{-\frac{1}{\sigma}} * \frac{\sigma}{\sigma +1 } \right] \left[  D^{*~\frac{\sigma+1}{\sigma}} - (\tau * D^*)^ \frac{\sigma+1}{\sigma} \right] - (1- \tau)*D_t^* * P_t^*$

### Balance equation ###
PROD_forLocal (sl) + PROD_forExp (sa) + IMP (da) + sum_reg(reg_trade_in) = CONS_fromLocal (dl) + CONS_fromImp (da) + EXP (sa) + sum_reg(reg_trade_out)

(Note that this means that sl includes alread reg_trade_out, and dl includes already reg_trade_in)


In [23]:
# Computing 
gamma = 0.5
trs   = 0.1


# supplies and demands in Mm^3, prices in €/m^3
sigma =        productDataReg['value','sigma']
pc    =        productDataReg['value','pc']
pc_l  = lagged_productDataReg['value','pc']
sc    =        productDataReg['value','sc']*1000000
sc_l  = lagged_productDataReg['value','sc']*1000000
dc    =        productDataReg['value','dc']*1000000
dc_l  = lagged_productDataReg['value','dc']*1000000
inv   =        productDataReg['value','in']*1000000
inv_l = lagged_productDataReg['value','in']*1000000


productDataReg['value','surpl_pr']  =  (sc * pc) - \
                                      (pc_l /  pow(sc_l,1/sigma)) \
                                    * pow(inv_l/inv,gamma/sigma) \
                                    * pow(sc,(1+sigma)/sigma) \
                                    * (sigma / (1+sigma))

productDataReg['value','surpl_co']  =    (pc_l*pow(dc_l,-1/sigma)*sigma/(sigma+1)) \
                                      * (pow(dc,(sigma+1)/sigma)-pow(trs*dc,(sigma+1)/sigma)) \
                                      - (1-trs) * dc * pc
        
productDataReg['value','surpl_tot'] =  productDataReg.fillna(0)['value','surpl_pr']+ productDataReg.fillna(0)['value','surpl_co']        
        
productDataReg[90:110]


value                      \
parName                                               dl        da        dt   
scen     country region prod      prodLev year                                 
P0-Mi-Ei 11000   11010  ashPlyW   secPr   2095  0.005935  0.008194  0.014129   
                                          2096  0.005936  0.008194  0.014130   
                                          2097  0.005937  0.008194  0.014131   
                                          2098  0.005937  0.008194  0.014132   
                                          2099  0.005938  0.008195  0.014132   
                                          2100  0.005938  0.008195  0.014133   
                                          2101  0.005938  0.008195  0.014133   
                                          2102  0.005938  0.008195  0.014133   
                                          2103  0.005938  0.008195  0.014134   
                                          2104  0.005938  0.008195  0.014134   
                                          2105  0.005939  0.008196  0.014134   
                                          2106  0.005939  0.008196  0.014135   
                                          2107  0.005939  0.008196  0.014135   
                        ashRoundW primPr  2005  0.000303  0.000000  0.000303   
                                          2006  0.000303  0.000000  0.000303   
                                          2007  0.000534  0.000000  0.000534   
                                          2008  0.000587  0.000000  0.000587   
                                          2009  0.000646  0.000000  0.000646   
                                          2010  0.000711  0.000000  0.000711   
                                          2011  0.000782  0.000000  0.000782   

                                                                              \
parName                                               dc        sl        sa   
scen     country region prod      prodLev year                                 
P0-Mi-Ei 11000   11010  ashPlyW   secPr   2095  0.007402  0.000000  0.000000   
                                          2096  0.007403  0.000000  0.000000   
                                          2097  0.007403  0.000000  0.000000   
                                          2098  0.007403  0.000000  0.000000   
                                          2099  0.007404  0.000000  0.000000   
                                          2100  0.007404  0.000000  0.000000   
                                          2101  0.007404  0.000000  0.000000   
                                          2102  0.007404  0.000000  0.000000   
                                          2103  0.007404  0.000000  0.000000   
                                          2104  0.007405  0.000000  0.000000   
                                          2105  0.007405  0.000000  0.000000   
                                          2106  0.007405  0.000000  0.000000   
                                          2107  0.007405  0.000000  0.000000   
                        ashRoundW primPr  2005  0.000000  0.012059  0.002225   
                                          2006  0.000000  0.012059  0.002225   
                                          2007  0.000000  0.011299  0.002398   
                                          2008  0.000000  0.011272  0.002448   
                                          2009  0.000000  0.011103  0.001911   
                                          2010  0.000000  0.011044  0.002014   
                                          2011  0.000000  0.011056  0.002271   

                                                                              \
parName                                               st        sc        pl   
scen     country region prod      prodLev year                                 
P0-Mi-Ei 11000   11010  ashPlyW   secPr   2095  0.000000  0.000000  609.1190   
                                          2096  0

In [29]:
productDataReg_byprodLev = productDataReg.groupby(level=['scen','country','region','prodLev','year']).sum() # attenction, everything is just summed, prices are obviously not correct
productDataReg_byprodLev[:130]

#productDataReg['value']=999

#test
#print(productDataReg[:500])
#st_reg = productDataReg[productDataReg['parName']=='st']
#st_reg = st_reg.drop(['freeDim'],axis=1) 
#st_reg.rename(columns={'value':'st_value'}, inplace=True)
#st_reg.set_index(['scen','country','region','prod','year','parName'], inplace=True)
#result = pd.concat([productDataNat, st_reg], axis=1)
#productDataNat = productDataReg.drop(['freeDim'],axis=1)    # we don't want this in national data, but still keep it in regional one
#productDataNat = productDataNat[productDataNat['parName'] != 'rt'] # we don't want rt in national data
#productDataNat.set_index(['scen','country','region','prod','year','parName'], inplace=True)
#productDataNat[:5]

value                       \
parName                                      dl        da         dt   
scen     country region prodLev year                                   
P0-Mi-Ei 11000   11010  primPr  2005   1.177444  0.000000   1.177444   
                                2006   1.177444  0.000000   1.177444   
                                2007   2.072304  0.000000   2.072304   
                                2008   2.279538  0.000000   2.279538   
                                2009   2.507488  0.000000   2.507488   
                                2010   2.756074  0.000000   2.756074   
                                2011   3.021258  0.000000   3.021258   
                                2012   3.103704  0.000000   3.103704   
                                2013   3.179758  0.000000   3.179758   
                                2014   3.190169  0.000000   3.190169   
                                2015   3.196658  0.000000   3.196658   
                                2016   3.203185  0.000000   3.203185   
                                2017   3.209888  0.000000   3.209888   
                                2018   3.208590  0.000000   3.208590   
                                2019   3.153060  0.000000   3.153060   
                                2020   3.159527  0.000000   3.159527   
                                2021   3.166259  0.000000   3.166259   
                                2022   3.173299  0.000000   3.173299   
                                2023   3.180579  0.000000   3.180579   
                                2024   3.187757  0.000000   3.187757   
                                2025   3.184899  0.000000   3.184899   
                                2026   3.183531  0.000000   3.183531   
                                2027   3.193019  0.000000   3.193019   
                                2028   3.203112  0.000000   3.203112   
                                2029   3.213418  0.000000   3.213418   
                                2030   3.224453  0.000000   3.224453   
                                2031   3.236661  0.000000   3.236661   
                                2032   3.240777  0.000000   3.240777   
                                2033   3.259108  0.000000   3.259108   
                                2034   3.269644  0.000000   3.269644   
...                                         ...       ...        ...   
                                2105   3.691337  0.000000   3.691337   
                                2106   3.697957  0.000000   3.697957   
                                2107   3.704523  0.000000   3.704523   
                        secPr   2005  11.564165  3.232008  14.796165   
                                2006  11.564165  3.232008  14.796165   
                                2007  10.374898  2.976182  13.351078   
                                2008  10.436372  2.921751  13.358129   
                                2009  10.272159  3.329893  13.602057   
                                2010  10.437677  3.025176  13.462849   
                                2011  10.499678  2.920586  13.420257   
                                2012  10.404306  3.154624  13.558924   
                                2013  10.443275  3.100023  13.543289   
                                2014  10.470362  3.098329  13.568687   
                                2015  10.497954  3.096737  13.594700   
                                2016  10.524705  3.095248  13.619958   
                                2017  10.550654  3.093870  13.644524   
                                2018  10.573941  3.092544  13.666490   
                                2019  10.583601  3.091309  13.674900   
                                2020  10.607092  3.090219  13.697319   
                                2021  10.629869  3.089224  13.719091   
                                2022  10.651944  3.088311  13.740269   
                                2023  10.673497  3.087395  13.760895   
          

## Saving dataframes to csv

In [32]:
productDataReg_byprodLev.to_csv("output/productDataReg_byprodLev.csv")